# Day 3 — corpus run

Attribute all 255 corpus prompts and record metrics. Before the loop, three checks that were
pre-registered on 22-09-2026 and whose outcomes decide how the loop's output is used.

| Part | What | Pre-registration |
|---|---|---|
| 1–2 | Environment, inputs, model | — |
| 3 | Loop machinery: one function per prompt, resumable JSONL writer | D3.3 failure handling |
| 4a | Fast metrics must reproduce the library; time the library function; new reference | D3.1, D3.2 |
| 4b | Layer-major reshape re-confirmed under the corrected code | D3.10 |
| 4c | ReplacementModel vs Day 2's HF model: top-1 agreement on 24 prompts | D3.4 |
| 5 | **Corpus loop**, resumable | D3.3 |
| 6 | Library pruning curve on a 32-graph subsample | D3.5 |
| 7 | Operational summary and save | D3.3 |

**Runtime.** ~24 s per graph → ~1.7 h for the corpus, ~15 min of checks, ~75 min for the pruning
subsample. Well inside the week's 30 GPU hours.

**Inputs.** Upload these three files as a Kaggle Dataset (Add Input → Upload) before starting:
`ct_utils.py` (the 22-09 version with `validate_against_library` and `err_final_share`),
`corpus.csv` from Day 2, and optionally `day1_config.json`.

**Discipline.** This notebook prints operational numbers only — statuses, timings, check results.
It deliberately does not print metric distributions by category. Those are for Day 6, and Day 5's
human ratings must be done blind to them.

Accelerator: **GPU T4 x2**.

## Part 0 — Install

Run first, before anything else is imported. (The earlier restart requirement came from installing
*after* `transformers` had been imported. As the very first cell, no restart is needed.)

In [1]:
!pip install -q circuit-tracer
print("installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.2/153.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.5/272.5 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 968.6/968.6 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.5/82.5 kB 2.9 MB/s eta 0:00:00
installed


## Part 1 — Environment

In [2]:
import os
os.environ["HF_HOME"] = "/kaggle/working/hf"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import sys, time, gc, json, glob, shutil, traceback
import numpy as np, pandas as pd, torch

cap = torch.cuda.get_device_capability(0)
GPU_NAME = torch.cuda.get_device_name(0)
assert cap >= (7, 0), f"{GPU_NAME} is sm_{cap[0]}{cap[1]}; P100 is unusable. Use T4 x2."
print(f"GPU: {GPU_NAME}  sm_{cap[0]}{cap[1]}")

OUT = "/kaggle/working/out"
os.makedirs(OUT, exist_ok=True)

def hw(peak=False):
    ram = os.popen("free -g | awk 'NR==2{print $7}'").read().strip() or "?"
    disk = os.popen("df -BG /kaggle/working | awk 'NR==2{print $4}'").read().strip()
    a = torch.cuda.memory_allocated() / 1e9
    t = torch.cuda.get_device_properties(0).total_memory / 1e9
    s = f"RAM free {ram} GB | GPU {a:.1f}/{t:.1f} GB | disk free {disk}"
    if peak:
        s += f" | peak {torch.cuda.max_memory_allocated()/1e9:.1f} GB"
    print(s)

def free():
    gc.collect(); torch.cuda.empty_cache()

hw()

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
from kaggle_secrets import UserSecretsClient
_tok = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = _tok
os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
from huggingface_hub import login, whoami
login(token=_tok)
print("logged in as:", whoami()["name"])

from importlib.metadata import version, PackageNotFoundError
def ver(d):
    try: return version(d)
    except PackageNotFoundError: return "unknown"
VERSIONS = {"torch": torch.__version__, "transformers": ver("transformers"),
            "circuit_tracer": ver("circuit-tracer"), "nnsight": ver("nnsight"),
            "numpy": ver("numpy"), "gpu": GPU_NAME}
print(VERSIONS)

## Part 2 — Inputs and model

In [ ]:
def find_input(name, required=True):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(f"/kaggle/working/{name}")
    if not hits:
        if required:
            raise FileNotFoundError(f"{name} not found. Add Input > Upload a dataset containing it.")
        return None
    return hits[0]

src = find_input("ct_utils.py")
shutil.copy(src, "/kaggle/working/ct_utils.py")
code = open("/kaggle/working/ct_utils.py").read()
assert "validate_against_library" in code and "err_final_share" in code, \
    "ct_utils.py is an OLD version - upload the 22-09 version"
sys.path.insert(0, "/kaggle/working")
import importlib, ct_utils
importlib.reload(ct_utils)
from ct_utils import graph_metrics_fast, validate_against_library, pruning_curve, to_np
print("ct_utils:", src)

corpus = pd.read_csv(find_input("corpus.csv"),
                     dtype={"expected": str, "top1_token": str, "subtype": str, "prompt": str},
                     keep_default_na=False)
print(f"corpus: {len(corpus)} prompts, {corpus.category.nunique()} categories")
print(corpus.groupby("category").size().to_string())
assert len(corpus) == 255 and corpus.prompt_id.is_unique

In [ ]:
from circuit_tracer import ReplacementModel, attribute
import circuit_tracer.graph as ctg

t0 = time.time()
model = ReplacementModel.from_pretrained(
    "google/gemma-2-2b", "gemma", backend="nnsight",
    dtype=torch.bfloat16, device="cuda", lazy_encoder=True,
)
print(f"loaded in {time.time()-t0:.0f}s")
free(); hw()

cfg_path = find_input("day1_config.json", required=False)
CONFIG = json.load(open(cfg_path)) if cfg_path else {}
CONFIG.update(
    config_version="day3",
    n_layers=model.cfg.n_layers,
    attr_kwargs=dict(max_feature_nodes=32768, batch_size=32),
    tolerance=1e-4,
    metrics_impl="graph_metrics_fast (library layout and definitions, 22-09-2026)",
    position_summaries="positions 1..n-1 (BOS excluded only; D3.10 branch 2)",
    versions=VERSIONS,
)
AK = CONFIG["attr_kwargs"]
LIB_FN = ctg.compute_graph_scores
print(json.dumps({k: v for k, v in CONFIG.items() if k != "versions"}, indent=2, default=str))

## Part 3 — Loop machinery

`process_prompt` handles one prompt end to end and **always returns a row**, implementing D3.3:

| outcome | `status` |
|---|---|
| normal | `ok` |
| feature cap saturated (guard raises) | `saturated` |
| out of GPU memory in attribution | `oom_attr` |
| out of GPU memory in metrics, CPU retry succeeds | `ok`, with `metrics_device = cpu` |
| CPU retry also fails | `oom_metrics` |
| BOS error > 1e-6 | `structural_zero_violated` (metrics kept; excluded from position summaries) |
| anything else | `error`, with message |

Each row also records the ReplacementModel's own top-1 token and probability, read from the graph.
That makes D3.4's recomputation free for every prompt.

In [ ]:
RESULTS = f"{OUT}/results.jsonl"
BOS_TOL = 1e-6
_printed_logit_attr = False

def graph_top1(g):
    '''ReplacementModel's top-1 token id and probability, read from the graph's logit nodes.'''
    global _printed_logit_attr
    probs = to_np(g.logit_probabilities)
    k = int(np.argmax(probs))
    for name in ("logit_tokens", "logit_targets"):
        if hasattr(g, name):
            seq = getattr(g, name)
            item = seq[k]
            if not _printed_logit_attr:
                print(f"[logit attribute: {name}, element type {type(item).__name__}]")
                _printed_logit_attr = True
            if torch.is_tensor(item) or isinstance(item, (int, np.integer)):
                return int(item), float(probs[k])
            for a in ("token_id", "vocab_idx", "token", "id"):
                val = getattr(item, a, None)
                if isinstance(val, (int, np.integer)) or torch.is_tensor(val):
                    return int(val), float(probs[k])
    raise AttributeError("could not read top-1 token from the graph; inspect vars(g)")

def process_prompt(row, keep_graph=False):
    rec = dict(prompt_id=row.prompt_id, category=row.category, subtype=row.subtype,
               n_tok=int(row.n_tok), status="ok", error_msg="", metrics_device="cuda")
    g = None
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    try:
        g = attribute(prompt=row.prompt, model=model, **AK)
    except torch.cuda.OutOfMemoryError as ex:
        rec.update(status="oom_attr", error_msg=str(ex)[:200]); free(); return rec, None
    except Exception as ex:
        rec.update(status="error", error_msg=f"attribute: {type(ex).__name__}: {str(ex)[:200]}")
        free(); return rec, None
    rec["attr_s"] = round(time.time() - t0, 2)

    try:
        tid, tp = graph_top1(g)
        rec["rm_top1_token"] = model.tokenizer.decode([tid])
        rec["rm_top1_prob"] = tp
    except Exception as ex:
        rec["rm_top1_token"], rec["rm_top1_prob"] = None, None
        rec["error_msg"] = f"top1: {str(ex)[:120]}"

    t0 = time.time()
    m = None
    try:
        m = graph_metrics_fast(g, CONFIG)
    except torch.cuda.OutOfMemoryError:
        free()
        try:
            m = graph_metrics_fast(g, CONFIG, device="cpu")
            rec["metrics_device"] = "cpu"
        except Exception as ex:
            rec.update(status="oom_metrics", error_msg=str(ex)[:200])
    except ValueError as ex:
        msg = str(ex)
        rec.update(status="saturated" if "saturated" in msg else "error", error_msg=msg[:200])
    except Exception as ex:
        rec.update(status="error", error_msg=f"metrics: {type(ex).__name__}: {str(ex)[:200]}")
    rec["metric_s"] = round(time.time() - t0, 2)
    rec["peak_gb"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)

    if m is not None:
        for k, v in m.items():
            rec[k] = json.dumps(v) if isinstance(v, (list, dict)) else v
        if abs(m["err_bos_pos"]) > BOS_TOL:
            rec["status"] = "structural_zero_violated"

    if not keep_graph:
        del g; g = None
    free()
    return rec, g

def append(rec):
    with open(RESULTS, "a") as f:
        f.write(json.dumps(rec, default=str) + "\n")
        f.flush(); os.fsync(f.fileno())

def done_ids():
    if not os.path.exists(RESULTS):
        return set()
    with open(RESULTS) as f:
        return {json.loads(l)["prompt_id"] for l in f if l.strip()}

print("machinery ready | already done:", len(done_ids()))

## Part 4a — Fast path vs the library (D3.1, D3.2)

`graph_metrics_fast` was corrected on 22-09 to use the library's node layout and completeness
definition. It must reproduce `compute_graph_scores` within 1e-4 on the reference prompt **and**
on the longest prompt in the corpus. The library function is also timed, for the D3.1 record.

In [ ]:
g_ref = attribute(prompt="The capital of France is", model=model, **AK)
LIB_FN(g_ref)                                    # warm-up
t0 = time.time(); LIB_FN(g_ref); lib_t_ref = time.time() - t0
print(f"library function on reference graph: {lib_t_ref:.2f}s")
m_ref = validate_against_library(g_ref, CONFIG, LIB_FN)

longest = corpus.sort_values("n_tok").iloc[-1]
print(f"\nlongest corpus prompt ({longest.n_tok} tokens, {longest.category}): {longest.prompt!r}")
g_big = attribute(prompt=longest.prompt, model=model, **AK)
t0 = time.time(); LIB_FN(g_big); lib_t_big = time.time() - t0
print(f"library function on longest graph: {lib_t_big:.2f}s")
m_big = validate_against_library(g_big, CONFIG, LIB_FN)
del g_big; free()

json.dump(m_ref, open(f"{OUT}/reference_france.json", "w"), indent=2)
CONFIG["d31"] = dict(library_seconds_reference=lib_t_ref, library_seconds_longest=lib_t_big,
                     reference_replacement=m_ref["replacement_score"],
                     reference_completeness=m_ref["completeness_score"])
print(f"\nnew reference: replacement {m_ref['replacement_score']:.4f}, "
      f"completeness {m_ref['completeness_score']:.4f}  (expected replacement ~0.7196)")
print(f"BOS {m_ref['err_bos_pos']:.6f} | final {m_ref['err_final_pos']:.4f} | "
      f"final share {m_ref['err_final_share']:.2f}")

## Part 4b — Layer-major reshape re-confirmed (D3.10)

Day 1 inferred layer-major storage from two buckets of zeros (BOS and final). Under the corrected
code only BOS is zero, so expect **one bucket of 26 at residue 0** under mod n_tok, and a flat
spread under mod 26.

In [ ]:
n_feat, nt, nl = len(g_ref.selected_features), len(to_np(g_ref.input_tokens)), CONFIG["n_layers"]
A = g_ref.adjacency_matrix.float().abs()
A = A / A.sum(1, keepdim=True).clamp_min(1e-30)
w = torch.zeros(A.shape[0], device=A.device)
w[-len(g_ref.logit_probabilities):] = g_ref.logit_probabilities.float().to(A.device)
v, inf = w.clone(), torch.zeros_like(w)
while float(v.sum()) > 1e-12:
    v = v @ A; inf += v
err_inf = inf[n_feat:n_feat + nl * nt].cpu().numpy()
del A, w, v, inf, g_ref; free()

z = np.where(err_inf == 0)[0]
by_nt = np.bincount(z % nt, minlength=nt)
by_26 = np.bincount(z % 26, minlength=26)
print(f"{len(z)} exact zeros of {len(err_inf)}")
print(f"mod {nt}:", by_nt)
print("mod 26:", by_26)
LAYER_MAJOR_OK = by_nt[0] == nl and by_nt[1:].sum() <= 2
print(f"\nlayer-major confirmed: {LAYER_MAJOR_OK}")
CONFIG["d310_layer_major"] = dict(zeros=int(len(z)), mod_ntok=by_nt.tolist(), confirmed=bool(LAYER_MAJOR_OK))
assert LAYER_MAJOR_OK, "zero pattern does not match layer-major storage - stop and inspect"


## Part 4c — Model consistency (D3.4)

Day 2's cheap predictors, `task_ok` and the `Fact:` decision came from an eager-attention HF model;
attribution runs on the nnsight ReplacementModel. 24 prompts (3 per category, `random_state=0`) are
attributed first, and the graph's top-1 compared with Day 2's.

**Pre-registered action:** 2 or more disagreements out of 24 → use the ReplacementModel's values.

**Refinement, stated before the check runs.** Every corpus row records the ReplacementModel's
top-1 token and probability from the graph, so recomputing top-1 and `task_ok` from them is exact
and free. Next-token entropy is **kept from Day 2** in either case: the graph stores only the top
logits, and a practitioner's cheap predictor is computed on a standard model, not on the
attribution wrapper. Log this refinement before running the cell.

These 24 graphs are corpus prompts; their rows are written to the results file and the loop skips
them.

In [ ]:
sample = corpus.groupby("category", group_keys=False).sample(3, random_state=0)
done = done_ids()
rows = []
for _, r in sample.iterrows():
    if r.prompt_id in done:
        rec = next(json.loads(l) for l in open(RESULTS) if json.loads(l)["prompt_id"] == r.prompt_id)
    else:
        rec, _ = process_prompt(r)
        append(rec)
    rows.append(dict(category=r.category, prompt=r.prompt[:38], day2_hf=r.top1_token,
                     replacement_model=rec.get("rm_top1_token"), status=rec["status"]))

cc = pd.DataFrame(rows)
cc["agree"] = cc.day2_hf == cc.replacement_model
cc["agree_stripped"] = cc.day2_hf.str.strip() == cc.replacement_model.fillna("").str.strip()
print(cc.to_string(index=False))

N_DISAGREE = int((~cc.agree).sum())
D34_TRIGGERED = N_DISAGREE >= 2
print(f"\ndisagreements: {N_DISAGREE}/24 (exact), {int((~cc.agree_stripped).sum())}/24 ignoring whitespace")
print(f">>> D3.4 {'TRIGGERED: use ReplacementModel top-1 and task_ok' if D34_TRIGGERED else 'not triggered: Day 2 values stand'}")
CONFIG["d34"] = dict(n_disagree=N_DISAGREE, triggered=bool(D34_TRIGGERED))
cc.to_csv(f"{OUT}/check_model_consistency.csv", index=False)

## Part 5 — Corpus loop

Resumable: rerunning this cell skips every prompt already in `results.jsonl`, so a dropped session
loses at most one prompt. Rows are flushed to disk as they are written. Graphs are discarded after
metrics — a single graph saves at ~685 MB and the disk cannot hold the corpus.

Set `RETRY_FAILED = True` only to re-attempt rows whose status is not `ok`.

**If the session drops:** start a new one, run Parts 0–3 (install, environment, inputs and model,
machinery), then come straight back here. Part 4 does not need rerunning — its outputs are saved.

In [ ]:
RETRY_FAILED = False

done = done_ids()
if RETRY_FAILED and os.path.exists(RESULTS):
    keep = [json.loads(l) for l in open(RESULTS) if l.strip()]
    failed = {r["prompt_id"] for r in keep if r["status"] not in ("ok", "structural_zero_violated")}
    with open(RESULTS, "w") as f:
        for r in keep:
            if r["prompt_id"] not in failed:
                f.write(json.dumps(r, default=str) + "\n")
    done -= failed
    print(f"retrying {len(failed)} failed prompts")

todo = corpus[~corpus.prompt_id.isin(done)]
print(f"{len(done)} done, {len(todo)} to go")

t_start = time.time()
for i, (_, r) in enumerate(todo.iterrows(), 1):
    rec, _ = process_prompt(r)
    append(rec)
    if i % 10 == 0 or i == len(todo) or rec["status"] != "ok":
        el = time.time() - t_start
        eta = el / i * (len(todo) - i)
        flag = "" if rec["status"] == "ok" else f"   <-- {rec['status']}: {rec['error_msg'][:60]}"
        print(f"{i:3d}/{len(todo)}  {r.category:22s} {r.n_tok:2d}t  "
              f"{rec.get('attr_s', 0):5.1f}s  elapsed {el/60:5.1f}m  eta {eta/60:5.1f}m{flag}")

print(f"\nfinished. total in results: {len(done_ids())}/{len(corpus)}")
hw(peak=True)

## Part 6 — Library pruning curve on the subsample (D3.5)

32 graphs, 4 per category, `random_state=0`, from prompts with status `ok`. Each is re-attributed
(graph structure is exactly reproducible; only bf16 arithmetic varies at ~5e-5) and the library's
`prune_graph` curve recorded alongside `compute_graph_scores`. ~140 s per graph, ~75 min total.
Resumable. Can be run in a later session.

In [ ]:
PRUNE = f"{OUT}/pruning_subsample.jsonl"
res = pd.DataFrame([json.loads(l) for l in open(RESULTS) if l.strip()])
ok_ids = set(res.loc[res.status == "ok", "prompt_id"])
sub = corpus[corpus.prompt_id.isin(ok_ids)].groupby("category", group_keys=False).sample(4, random_state=0)
pdone = {json.loads(l)["prompt_id"] for l in open(PRUNE)} if os.path.exists(PRUNE) else set()
print(f"subsample {len(sub)} graphs, {len(pdone)} already done")

for i, (_, r) in enumerate(sub[~sub.prompt_id.isin(pdone)].iterrows(), 1):
    t0 = time.time()
    try:
        g = attribute(prompt=r.prompt, model=model, **AK)
        lib_rep, lib_comp = LIB_FN(g)
        curve = pruning_curve(g, ctg.prune_graph)
        rec = dict(prompt_id=r.prompt_id, category=r.category, lib_replacement=lib_rep,
                   lib_completeness=lib_comp, lib_prune_nodes=curve, status="ok")
        del g
    except Exception as ex:
        rec = dict(prompt_id=r.prompt_id, category=r.category, status="error",
                   error_msg=f"{type(ex).__name__}: {str(ex)[:200]}")
    free()
    with open(PRUNE, "a") as f:
        f.write(json.dumps(rec, default=str) + "\n"); f.flush(); os.fsync(f.fileno())
    print(f"{i:2d}  {r.category:22s} {time.time()-t0:6.1f}s  {rec['status']}")

In [ ]:
# Agreement between the cheap node-count curve and the library's pruning curve,
# and between the fast path and the library's scores, on the subsample.
ps = pd.DataFrame([json.loads(l) for l in open(PRUNE) if l.strip()])
ps = ps[ps.status == "ok"].merge(res[["prompt_id", "replacement_score", "completeness_score",
                                      "node_count_curve"]], on="prompt_id")
out_rows = []
for t in ("0.95", "0.9", "0.8", "0.7"):
    lib = ps.lib_prune_nodes.apply(lambda d: d.get(t) if isinstance(d, dict) else None).astype(float)
    ours = ps.node_count_curve.apply(lambda s: json.loads(s)[t]).astype(float)
    out_rows.append(dict(threshold=t, spearman=lib.corr(ours, method="spearman"), n=int(lib.notna().sum())))
agree = pd.DataFrame(out_rows)
print(agree.round(3).to_string(index=False))
print(f"\nmax |replacement - library| on subsample: {(ps.replacement_score - ps.lib_replacement).abs().max():.2e}")
print(f"max |completeness - library| on subsample: {(ps.completeness_score - ps.lib_completeness).abs().max():.2e}")
print("(differences here include re-attribution bf16 noise, ~5e-5)")
agree.to_csv(f"{OUT}/pruning_agreement.csv", index=False)

## Part 7 — Operational summary and save

Statuses by category, D3.3's 10% flag, timing, and the D3.4 outcome. **No outcome distributions** —
those are Day 6.

In [ ]:
res = pd.DataFrame([json.loads(l) for l in open(RESULTS) if l.strip()])
assert res.prompt_id.is_unique, "duplicate rows in results.jsonl"
missing = set(corpus.prompt_id) - set(res.prompt_id)
print(f"rows: {len(res)}/{len(corpus)}   missing: {len(missing)}")

st = pd.crosstab(res.category, res.status, margins=True)
print("\n" + st.to_string())

# structural_zero_violated rows keep their metrics (only position summaries drop them),
# so they do not count as lost for the D3.3 flag.
lost = res.groupby("category").status.apply(
    lambda s: (~s.isin(["ok", "structural_zero_violated"])).mean())
flagged = lost[lost > 0.10]
print(f"\ncategories losing >10% (D3.3 flag): {flagged.round(2).to_dict() or 'none'}")
print(f"BOS structural-zero violations: {(res.status == 'structural_zero_violated').sum()}")
print(f"metrics retried on CPU: {(res.metrics_device == 'cpu').sum()}")
ok = res[res.status == "ok"]
print(f"\nattribution: mean {ok.attr_s.mean():.1f}s, max {ok.attr_s.max():.1f}s | "
      f"peak GPU {res.peak_gb.max():.1f} GB")

In [ ]:
# D3.4 branch: if triggered, top-1 and task_ok come from the ReplacementModel for every prompt.
# Works in a fresh session too: falls back to the saved consistency check.
if "D34_TRIGGERED" not in globals():
    _cc = pd.read_csv(f"{OUT}/check_model_consistency.csv", keep_default_na=False)
    D34_TRIGGERED = int((_cc.day2_hf != _cc.replacement_model).sum()) >= 2
    print("D3.4 outcome reloaded from check_model_consistency.csv:", D34_TRIGGERED)

def answers(pred, expected):
    p = str(pred or "").strip().lower()
    return bool(p) and bool(expected) and str(expected).lower().startswith(p)

merged = corpus.merge(res[["prompt_id", "rm_top1_token", "rm_top1_prob"]], on="prompt_id", how="left")
merged["top1_agree"] = merged.top1_token == merged.rm_top1_token
print(f"top-1 agreement HF vs ReplacementModel, whole corpus: {merged.top1_agree.mean():.1%}")

if D34_TRIGGERED:
    ans_cats = ["factual_recall", "multi_hop", "arithmetic", "induction"]
    m = merged.category.isin(ans_cats) & (merged.expected != "")
    merged["task_ok_rm"] = None
    merged.loc[m, "task_ok_rm"] = merged.loc[m].apply(lambda r: answers(r.rm_top1_token, r.expected), axis=1)
    print("task_ok recomputed from ReplacementModel for:", ans_cats)
    print("syntax task_ok (verb-logit margin) retained from Day 2 - not recoverable from top logits.")
merged.to_csv(f"{OUT}/corpus_with_rm.csv", index=False)

In [ ]:
res.to_csv(f"{OUT}/results.csv", index=False)
CONFIG["run_summary"] = dict(
    rows=int(len(res)), status_counts=res.status.value_counts().to_dict(),
    categories_flagged_over_10pct=flagged.round(3).to_dict(),
    mean_attr_s=float(ok.attr_s.mean()), peak_gpu_gb=float(res.peak_gb.max()),
    corpus_top1_agreement=float(merged.top1_agree.mean()),
)
json.dump(CONFIG, open(f"{OUT}/day3_config.json", "w"), indent=2, default=str)
shutil.copy("/kaggle/working/ct_utils.py", f"{OUT}/ct_utils.py")
print(sorted(os.listdir(OUT)))
print("\n*** DOWNLOAD /kaggle/working/out/ NOW, then Save Version ***")

## Record in the pre-registration

Fill from `day3_config.json` and the outputs above.

> **D3.1 — outcome.** Library `compute_graph_scores` took [__] s on the reference graph and [__] s on
> the longest corpus prompt. `graph_metrics_fast` reproduced it within [__] on both. The library's
> definition is primary, computed via the fast path; `compute_graph_scores` was also computed on
> the 32-graph subsample, where the maximum difference was [__] (including re-attribution noise).
>
> **D3.4 refinement, recorded before the check.** Top-1 and `task_ok` are recomputed from the
> ReplacementModel's graph logits if triggered, which is exact and free; next-token entropy is kept
> from the Day 2 HF model, because the graph stores only the top logits and a practitioner's cheap
> predictor is computed on a standard model.
>
> **D3.4 — outcome.** [__]/24 disagreements → [not triggered / triggered]. Whole-corpus top-1
> agreement: [__]%.
>
> **D3.10 — layer-major re-confirmed.** [__] zeros, [26] at residue 0 mod n_tok, flat under mod 26.
>
> **Corpus run.** [__]/255 `ok`. Status counts: [__]. Categories losing >10%: [none / __]. BOS
> violations: [__]. Mean attribution [__] s; peak GPU [__] GB.
>
> **D3.5 — pruning subsample.** Spearman correlation between node-count curve and library pruning
> curve: [__] / [__] / [__] / [__] at 0.95 / 0.9 / 0.8 / 0.7.